In [8]:
from ..Tools import *

#多工具调用

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = os.getenv('DEEPSEEK_BASE_URL')
model = init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek-v4-flash',
    model_provider='deepseek'
    #extra_body={"thinking":{"type":"enabled"}
)


#使用BaseModel定义pydantic参数模型
class weatherInput(BaseModel):
    city: str = Field(
        default='成都',
        description='城市'
    )
    #Literal固定参数值,类似枚举
    unit: Literal['celsius', 'fahrenheit'] = Field(
        default='celsius',
        description='温度单位'
    )
    isFuture: bool = Field(
        default=False,
        description='是否查询未来五天的天气'
    )


@tool(description='查询天气', args_schema=weatherInput)
def get_weather(city: str, unit: str, isFuture: bool) -> str:
    temp = 22 if unit == 'celsius' else 72
    result = f'{city}温度为：{temp}{"摄氏度" if unit == "celsius" else "华氏度."}'
    if isFuture:
        result += '未来五天温度保持不变.'
    return result


@tool(parse_docstring=True)
def search_news(content:Optional[str]=None, location: str = '成都') -> str:
    """
    获取当地今日新闻

    Args:
        location:地区
        content:新闻内容
    """
    if content is None:
        result = f'{location}今日暂无新闻'
    else:
        result = f'{location}今日最新新闻为：{content}'
    return result

In [13]:
print('=' * 5, "欢迎使用天气与新闻查询工具", '=' * 5, end='\n')
location = input('请输入查询地点：')
message_list = [
    SystemMessage('你是小灵，一个可靠的AI辅助工具')
]
message_list.append(HumanMessage(f'我需要查询{location}的今日天气和新闻'))
#bind_tools方法可以传入tool_choice参数来决定模型调用工具的方式
#none表示不调用工具，auto表示模型自动决定是否调用，required表示不管提示词如何，都必须调用工具，同时也可以指定工具名调用特定工具
model_with_tools = model.bind_tools([get_weather, search_news],
                                    # tool_choice="none"
                                    # tool_choice="auto"
                                    # tool_choice="required"
                                    # tool_choice="search_news"
                                    )
response = model_with_tools.invoke(message_list)
# print(response)
tool_calls = response.tool_calls
for tool_call in tool_calls:
    if tool_call['name'] == 'get_weather':
        print('返回天气工具调用结果', end='\n')
        tool_msg = get_weather.invoke(tool_call)
        message_list.append(tool_msg)
    if tool_call['name'] == 'search_news':
        print('返回新闻工具调用结果', end='\n')
        tool_msg = search_news.invoke(tool_call)
        message_list.append(tool_msg)
for msg in message_list:
    msg.pretty_print()

===== 欢迎使用天气与新闻查询工具 =====
返回天气工具调用结果
返回新闻工具调用结果
================================ System Message ================================

你是小灵，一个可靠的AI辅助工具
================================ Human Message =================================

我需要查询北京的今日天气和新闻
================================= Tool Message =================================
Name: get_weather

北京温度为：22摄氏度
================================= Tool Message =================================
Name: search_news

北京今日暂无新闻
